In [ ]:
pip install biopython freesasa numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.6 MB/s eta 0:00:00
  Created wheel for freesasa: filename=freesasa-2.2.1-cp312-cp312-linux_x86_64.whl size=951906 sha256=3a2608b56a7fac4a6c66e2ad2626892317ea47e5c73dbabe7acc516087896869
  Stored in directory: /root/.cache/pip/wheels/d5/ba/ad/c7be874883ac321912d0442a2e8ff4208d0c3bf3bf94952387
Successfully built freesasa


In [ ]:
import freesasa
from Bio.PDB import PDBParser, PDBIO, NeighborSearch, Selection

# INPUT
PDB_FILE = "/content/1HZH.pdb"
DISTANCE_CUTOFF = 5.0  # Å  # standard threshold to define interface residues

# interacting chains (antibody–antigen).
CHAIN_A_ID = "H"
CHAIN_B_ID = "K"

# 1. LOAD STRUCTURE
# Reads the PDB file & Extracts the two interacting chains
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", PDB_FILE)
model = structure[0]

chain_A = model[CHAIN_A_ID]
chain_B = model[CHAIN_B_ID]

# 2. INTERFACE RESIDUES
# Converts each chain into a flat list of atoms
atoms_A = Selection.unfold_entities(chain_A, "A")
atoms_B = Selection.unfold_entities(chain_B, "A")

# Builds a spatial tree for fast neighbor queries
ns = NeighborSearch(atoms_A + atoms_B)

interface_A = set()
interface_B = set()

'''
For every atom in chain A:
Find atoms within 5 Å.
If any belong to chain B → that residue is interface residue.
'''
for atom in atoms_A:
    for nb in ns.search(atom.coord, DISTANCE_CUTOFF):
        if nb.get_parent().get_parent().id == CHAIN_B_ID:
            interface_A.add(atom.get_parent())

for atom in atoms_B:
    for nb in ns.search(atom.coord, DISTANCE_CUTOFF):
        if nb.get_parent().get_parent().id == CHAIN_A_ID:
            interface_B.add(atom.get_parent())

print("Interface residues (≤ 5 Å):")
print(f"Chain {CHAIN_A_ID}: {[r.get_id()[1] for r in interface_A]}")
print(f"Chain {CHAIN_B_ID}: {[r.get_id()[1] for r in interface_B]}")


# 3. WRITE SINGLE-CHAIN PDB FILES
io = PDBIO()

def write_chain(chain, filename):
    io.set_structure(chain)
    io.save(filename)

pdb_A = "chain_H.pdb"
pdb_B = "chain_K.pdb"


write_chain(chain_A, pdb_A)
write_chain(chain_B, pdb_B)

'''
SASA "Solvent Accessible Surface Area", SASA tells you how much of the protein is exposed to water.
BSA  "Buried Surface Area", surface area that becomes hidden when two proteins bind together
'''
# 4. CALCULATE SASA AND BSA
sasa_complex = freesasa.calc(freesasa.Structure(PDB_FILE)).totalArea()
sasa_A = freesasa.calc(freesasa.Structure(pdb_A)).totalArea()
sasa_B = freesasa.calc(freesasa.Structure(pdb_B)).totalArea()

BSA = (sasa_A + sasa_B) - sasa_complex

print("\nSurface Area Results (Å²):")
print(f"SASA Chain {CHAIN_A_ID}: {sasa_A:.2f}")
print(f"SASA Chain {CHAIN_B_ID}: {sasa_B:.2f}")
print(f"SASA Complex: {sasa_complex:.2f}")
print(f"Buried Surface Area (BSA): {BSA:.2f}")


# 5. HOTSPOT PREDICTION (ΔSASA PROXY)
# Residues with largest interface presence are likely hotspots

hotspot_candidates = []

for residue in interface_A.union(interface_B):  # Only residues actually touching the partner are considered
    resname = residue.get_resname()
    resid = residue.get_id()[1]
    chain = residue.get_parent().id
    num_atoms = len(Selection.unfold_entities(residue, "A"))  # Residues with more atoms at the interface are ranked higher
    hotspot_candidates.append((chain, resname, resid, num_atoms))

# Sort by atom count (proxy for ΔSASA contribution)
hotspots = sorted(hotspot_candidates, key=lambda x: x[3], reverse=True)[:5]

print("\nTop 5 Predicted Hotspot Residues:")
for chain, resname, resid, score in hotspots:
    print(f"Chain {chain} - {resname} {resid} | Interface atom count: {score}")


Interface residues (≤ 5 Å):
Chain H: [491, 240, 389, 237, 373, 388, 314, 372, 425, 236, 378, 391, 500, 375, 440, 422, 428, 239, 387, 427, 493, 418, 421, 374, 368, 470, 436, 420, 238, 393, 235, 377, 439, 383, 438, 371, 370, 423, 247, 376, 475, 426]
Chain K: [439, 371, 383, 511, 244, 495, 370, 514, 498, 423, 247, 438, 376, 389, 475, 426, 491, 240, 373, 246, 425, 388, 230, 391, 372, 440, 422, 378, 375, 428, 239, 387, 427, 493, 418, 368, 248, 421, 232, 374, 470, 436, 420, 442, 393, 235, 377]

Surface Area Results (Å²):
SASA Chain H: 26939.17
SASA Chain K: 26025.70
SASA Complex: 65501.01
Buried Surface Area (BSA): -12536.14

Top 5 Predicted Hotspot Residues:
Chain K - TYR 438 | Interface atom count: 12
Chain H - TYR 438 | Interface atom count: 12
Chain K - TYR 370 | Interface atom count: 12
Chain H - TYR 370 | Interface atom count: 12
Chain K - ARG 376 | Interface atom count: 11
